In [1]:
import os
import tensorflow as tf
import cv2
import numpy as np
import time
import csv
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as viz_utils

INPUT_VIDEO = r"D:\Documents\University\Adv Proj\Vdo-Demo-Tester\demo1(dawn).mp4"
OUTPUT_VIDEO = r"D:\Documents\University\Adv Proj\output_demo\Demo3\SSD\processed_ssd.mp4"
CSV_LOG_PATH = r"D:\Documents\University\Adv Proj\output_demo\Demo3\SSD\ssd_detection_log.csv"
PATH_TO_SAVED_MODEL = r"D:\Documents\University\Adv Proj\Model\tf_workspace\exported-models\my_mobnet_model\saved_model"
PATH_TO_LABELS = r"D:\Documents\University\Adv Proj\Model\tf_workspace\annotations\label_map.pbtxt"

os.environ['XLA_FLAGS'] = r"--xla_gpu_cuda_data_dir=D:\Terminal\ancondaEnviroment\driver_ai\Library"
os.makedirs(os.path.dirname(CSV_LOG_PATH), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_VIDEO), exist_ok=True)

print("Loading SSD model")
detect_fn = tf.saved_model.load(PATH_TO_SAVED_MODEL)
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

cap = cv2.VideoCapture(INPUT_VIDEO)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Frame_ID", "Detection_Class", "Confidence_Score", "Inference_Time_ms"])

    frame_id = 0
    print("Processing video")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        input_tensor = tf.convert_to_tensor(frame)[tf.newaxis, ...]

        start = time.perf_counter()
        detections = detect_fn(input_tensor)
        end = time.perf_counter()

        latency = (end - start) * 1000

        scores = detections['detection_scores'][0].numpy()
        classes = detections['detection_classes'][0].numpy()
        
        best_conf = float(scores[0])
        class_id = int(classes[0]) + 1
        
        if class_id in category_index:
            best_class = category_index[class_id]['name']
        else:
            best_class = "None"

        writer.writerow([frame_id, best_class, round(best_conf, 4), round(latency, 2)])

        viz_utils.visualize_boxes_and_labels_on_image_array(
              frame,
              detections['detection_boxes'][0].numpy(),
              (classes + 1).astype(int),
              scores,
              category_index,
              use_normalized_coordinates=True,
              min_score_thresh=.50)

        out.write(frame)
        frame_id += 1

cap.release()
out.release()
print("Done")

d:\Terminal\ancondaEnviroment\driver_ai\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
d:\Terminal\ancondaEnviroment\driver_ai\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\Terminal\ancondaEnviroment\driver_ai\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-a

Loading SSD model
Processing video
Done
